In [2]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# 1. 首次聯網下載模型與 Tokenizer
model_name = "ProsusAI/finbert"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. 儲存到本地端資料夾 (例如命名為 local_finbert)
model.save_pretrained("./local_finbert")
tokenizer.save_pretrained("./local_finbert")

# ==========================================
# 之後你每次要測試時，只需要指向本地資料夾即可：
offline_nlp = pipeline("sentiment-analysis", model="./local_finbert", tokenizer="./local_finbert")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 33494.44it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10579.71it/s]


In [3]:
# 載入 FinBERT 模型 (專門針對金融語境訓練的預訓練模型)
# 實作時首次執行會自動從 Hugging Face 下載模型
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# 模擬從 Yahoo Finance API 或 News API 抓取到的近期新聞標題
news_headlines = [
    "Tech giant reports record-breaking quarterly revenue and strong guidance.",
    "Supply chain issues and regulatory headwinds might cause a slight delay.",
    "The overall market faces severe inflation fears and rising interest rates."
]

def calculate_sentiment_score(headlines):
    results = nlp(headlines)
    # 將標籤轉為數值權重
    score_mapping = {"positive": 1, "neutral": 0, "negative": -1}
    total_score = 0
    
    for res in results:
        label = res['label']
        confidence = res['score'] # 模型對該情緒的信心水準
        # 加權情緒分數
        total_score += score_mapping[label] * confidence
        
    # 計算平均市場情緒分數，數值介於 -1 到 1 之間
    avg_score = total_score / len(headlines) if headlines else 0
    return avg_score

sentiment = calculate_sentiment_score(news_headlines)
print(f"該 ETF 近期市場情緒分數為: {sentiment:.2f} (正值為樂觀，負值為悲觀)")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 50262.63it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


該 ETF 近期市場情緒分數為: -0.21 (正值為樂觀，負值為悲觀)


In [4]:
!pip install yfinance --quiet
import yfinance as yf

In [ ]:
# 載入你已下載到本地端的模型
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

def get_etf_sentiment(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)
    news_list = ticker.news
    
    if not news_list:
        return 0.0, []
    
    # 【除錯步驟】把第一筆新聞的完整結構印出來看看，這樣你就知道現在的 Key 叫什麼了
    print("【除錯資訊】目前 yfinance 回傳的一筆新聞結構如下：")
    print(news_list[0]) 
    print("-" * 50)
    
    headlines = []
    # 使用容錯寫法抓取標題
    for article in news_list:
        # 使用 .get()，如果沒有 'title' 這個 key，就回傳 None，避免程式崩潰
        # 有些版本可能藏在 'content' 裡面，所以多加一層判斷
        title = article.get('title') 
        
        if not title and 'content' in article:
            title = article['content'].get('title')
            
        if title: # 確定有抓到字串才加進去
            headlines.append(title)
            
    if not headlines:
        print("警告：無法從新聞資料中萃取出標題，請檢查資料結構。")
        return 0.0, []
        
    # 進行情緒推論
    results = nlp(headlines)
    score_mapping = {"positive": 1, "neutral": 0, "negative": -1}
    total_score = 0
    
    for res in results:
        total_score += score_mapping[res['label']] * res['score']
        
    avg_score = total_score / len(headlines)
    return avg_score, headlines

# 測試：抓取 SPDR S&P 500 ETF (SPY) 的即時情緒
spy_score, spy_news = get_etf_sentiment("SPY")
print(f"SPY 即時市場情緒分數: {spy_score:.2f}")
print("參考的新聞標題：", spy_news[:3]) # 印出前三筆看看

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30866.44it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40193.33it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


【除錯資訊】目前 yfinance 回傳的一筆新聞結構如下：
{'id': '8d98ee30-dc17-3c50-bcf6-c4f977483635', 'content': {'id': '8d98ee30-dc17-3c50-bcf6-c4f977483635', 'contentType': 'STORY', 'title': "Foreigners Own Nearly $30 Trillion in U.S. Stocks and Bonds. Here Is Why That Number Should Be on Every Investor's Radar", 'description': '', 'summary': "Foreign investor sentiment is a barometer you shouldn't overlook.", 'pubDate': '2026-04-11T18:05:00Z', 'displayTime': '2026-04-11T18:05:00Z', 'isHosted': True, 'bypassModal': False, 'previewUrl': None, 'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/motleyfool.com/db7bbf3b214bc5cb1ea7eece84c230d6', 'originalWidth': 1400, 'originalHeight': 933, 'caption': 'Three people sitting around a table, discussing paperwork.', 'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/_ky08aijQHeFudeHVT7YNQ--~B/aD05MzM7dz0xNDAwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/motleyfool.com/db7bbf3b214bc5cb1ea7eece84c230d6', 'width': 1400, 'height': 933, 'tag': 'original'}